# Carga y unificación de dataset

In [9]:
import pandas as pd
import glob
import os

# Define the path to the datasets directory
dataset_path = 'datasets'

# Get a list of all CSV files in the directory
csv_files = glob.glob(os.path.join(dataset_path, '*.csv'))

# Load all CSV files into a list of DataFrames
dfs = []
for filename in csv_files:
    df = pd.read_csv(filename)
    dfs.append(df)

# Concatenate all DataFrames into a single DataFrame
if dfs:
    full_df = pd.concat(dfs, ignore_index=True)
    
    # Remove the filename column if it exists
    if 'filename' in full_df.columns:
        full_df = full_df.drop(columns=['filename'])
    
    # Export the unified dataframe to a CSV file
    # full_df.to_csv('dataset_unified.csv', index=False)
    # print("Unified dataset exported to 'dataset_unified.csv'.")
    
    print("Dataset loaded successfully.")
    print(f"Total rows: {len(full_df)}")
    display(full_df.head())
else:
    print("No CSV files found in the datasets directory.")

Dataset loaded successfully.
Total rows: 6000


,person,audio,noise,snr,provider,text,status,transcription_time
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59
3,p7,1,clean,NaN,custom,Genera una cotización para el cliente con FooF...,success,1.66
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53


In [10]:
mean_time_by_provider = full_df.groupby('provider')['transcription_time'].mean()
print("Mean Transcription Time by Provider:")
print(mean_time_by_provider)

Mean Transcription Time by Provider:
provider
amazon    12.379000
azure      2.338153
custom     1.380573
google     1.011293
Name: transcription_time, dtype: float64


# Normalización

In [11]:
import sys
import os

# Add the tools directory to the system path
sys.path.append(os.path.abspath('tools'))
from tools.normalize_numeric import TextNormalizerNumeric

normalizer = TextNormalizerNumeric()


# Apply normalization to the text column
if 'text' in full_df.columns:
    full_df['text_normalized'] = full_df['text'].apply(normalizer.normalize)

    # Rellenar valores nulos en 'snr' con "clean"
    full_df['snr'] = full_df['snr'].fillna('clean')
    
    # Replace 'custom' with 'whisper' in 'provider' column
    if 'provider' in full_df.columns:
        full_df['provider'] = full_df['provider'].replace('custom', 'whisper')

    print("Text column normalized successfully.")
    display(full_df.head(3))
    
    full_df.to_csv('dataset_normalized.csv', index=False)
else:
    print("Text column not found in the dataframe.")

Text column normalized successfully.


,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized
0,p7,1,cafe,0dB,whisper,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...
1,p7,1,cafe,5dB,whisper,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...
2,p7,1,cafe,10dB,whisper,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...


In [13]:
filtered_df = full_df.loc[(full_df['person'] == 'p4') & (full_df['noise'] == 'clean') & (full_df['provider'] == 'whisper')]
display(filtered_df[['audio','text']].head(3))

,audio,text
5253,1,Genera una cotización para el cliente CompuFac...
5263,2,Prepara un presupuesto urgente con 10 teclados...
5273,3,Crea una oferta comercial para Carla Santana c...
